# 02 · Structured outputs

**Goal:** get data from Claude that your code can use directly, as a validated Python object instead of text you have to clean up.

| | |
|---|---|
| **You will learn** | Why plain JSON requests break, what happened to the prefill trick, structured outputs with Pydantic, raw JSON schemas, and when structured outputs can still fail |
| **Prerequisites** | [Notebook 00](00_setup_and_first_request.ipynb) and [notebook 01](01_conversations_and_control.ipynb) |
| **Run time** | About 1 minute |
| **Cost** | About $0.01 when I ran it. This notebook stops itself at $0.10 |
| **Paid cells** | Most cells make a small paid call. Section 2 is free, because the request is rejected |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Setup

The same setup as the earlier notebooks. Run the cells from top to bottom.

In [1]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [2]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.10     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

## 1. The problem: Claude is too helpful (paid)

Say you're building a small web app that generates AWS EventBridge rules. The user clicks a button and your code needs clean JSON. Claude, though, likes to add helpful extras: code fences around the JSON, and a sentence explaining it. A person doesn't mind. `json.loads()` does.

In [3]:
response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule as JSON"}],
))

text = text_from_message(response)
print("What came back:\n" + text + "\n")

try:
    json.loads(text)
    print("json.loads worked this time, but nothing promised it would.")
except json.JSONDecodeError as error:
    print(f"json.loads failed: {error}")

[input=25 output=239 cost=$0.00244 | spent $0.0024 of $0.10]
What came back:
```json
{
  "Name": "ec2-state-change-rule",
  "Description": "Trigger when an EC2 instance changes state",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["stopped", "terminated"]
    }
  },
  "State": "ENABLED"
}
```

This rule matches EC2 instances that enter the `stopped` or `terminated` state. You can create it with:

```bash
aws events put-rule --cli-input-json file://rule.json
```

Then attach a target (such as an SNS topic or Lambda function) with `aws events put-targets`.

json.loads failed: Expecting value: line 1 column 1 (char 0)


## 2. The course's trick no longer works (free)

The course solves this with **assistant prefill** plus a **stop sequence**:

1. Add a fake assistant message containing `` ```json ``, so Claude believes it already opened a code block.
2. Claude continues with only the JSON.
3. Set `` ``` `` as a stop sequence, so generation ends the moment Claude tries to close the block.

It was clever, and it still works on older models such as Haiku 4.5. **Current models, including Sonnet 5.5, reject prefill with a 400 error.** Let's see it. A rejected request generates nothing, so it isn't billed.

In [4]:
import anthropic

try:
    client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[
            {"role": "user", "content": "Generate a very short EventBridge rule as JSON"},
            {"role": "assistant", "content": "```json"},  # the prefill
        ],
        stop_sequences=["```"],
    )
    raise AssertionError("Expected a BadRequestError")
except anthropic.BadRequestError as error:
    print(f"BadRequestError, status {error.status_code}")
    print(error.message)

BadRequestError, status 400
Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CgjoKKTsVY1ktmmxFnzKy'}


## 3. A weak fix: ask nicely, then clean up (paid)

The simplest replacement is to ask for raw JSON in the prompt, then cut out whatever lies between the first `{` and the last `}`.

In [5]:
response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": (
        "Generate a very short EventBridge rule. Respond with ONLY raw JSON: no markdown fences, no explanation."
    )}],
))

text = text_from_message(response)
print("What came back:\n" + text + "\n")

start = text.find("{")
end = text.rfind("}")
rule = json.loads(text[start:end + 1])

print("Parsed:", rule)
assert isinstance(rule, dict)

[input=48 output=63 cost=$0.00073 | spent $0.0032 of $0.10]
What came back:
{"Name":"every-5-min-rule","ScheduleExpression":"rate(5 minutes)","State":"ENABLED","Description":"Triggers every 5 minutes"}

Parsed: {'Name': 'every-5-min-rule', 'ScheduleExpression': 'rate(5 minutes)', 'State': 'ENABLED', 'Description': 'Triggers every 5 minutes'}


**What to notice**

- This often works, but **a prompt is a request, not a guarantee**. Nothing checks that the keys you need exist, or that a value has the right type.
- The clean-up is fragile too. It breaks if the answer is a JSON array, or if Claude's explanation contains a `{`.

For one-off experiments that's fine. For anything your code depends on, use the next section.

## 4. The proper way: structured outputs (paid)

Think of the difference between asking someone to *write you a letter* with their details, and handing them a **form** with labelled boxes. The form decides the shape of the answer.

With structured outputs, you describe the shape as a Pydantic class and call `client.messages.parse()`. The SDK turns your class into a JSON schema and sends it, the API constrains Claude's output to fit it, and you get back a real Python object in `response.parsed_output`.

In [6]:
from pydantic import BaseModel


class EventBridgeRule(BaseModel):
    source: list[str]
    detail_type: list[str]   # simplified: real EventBridge rules call this "detail-type"
    description: str


response = track(client.messages.parse(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule for EC2 instances entering the running state."}],
    output_format=EventBridgeRule,
))

rule = response.parsed_output

print("Type       :", type(rule).__name__)
print("source     :", rule.source)
print("detail_type:", rule.detail_type)
print("description:", rule.description)
print("\nAs JSON for your app:", rule.model_dump_json())

assert isinstance(rule, EventBridgeRule)
assert isinstance(rule.source, list)

[input=348 output=60 cost=$0.00130 | spent $0.0045 of $0.10]
Type       : EventBridgeRule
source     : ['aws.ec2']
detail_type: ['EC2 Instance State-change Notification']
description: Triggers when an EC2 instance enters the running state.

As JSON for your app: {"source":["aws.ec2"],"detail_type":["EC2 Instance State-change Notification"],"description":"Triggers when an EC2 instance enters the running state."}


**What to notice**

- No clean-up. `rule.source` is already a Python list, and your editor can autocomplete it.
- **Compare `input` with sections 1 and 3.** When I ran this it jumped from 25 tokens to 348, because Claude also receives instructions describing the format. A bigger class costs more: the one in section 6 used 679. Keep your classes as small as the job allows.
- The first call with a new schema can take a little longer while the API compiles it. Compiled schemas are cached for 24 hours.

## 5. Without Pydantic: a raw JSON schema (paid)

`parse()` with Pydantic is the convenient route, but it's not required. You can pass a JSON schema yourself through `output_config.format` on the normal `create()` call. You get text back that's guaranteed to parse, and you call `json.loads()` on it yourself.

Two rules for raw schemas: every object needs `"additionalProperties": False`, and you list its fields in `required`.

In [7]:
rule_schema = {
    "type": "object",
    "properties": {
        "source": {"type": "array", "items": {"type": "string"}},
        "description": {"type": "string"},
    },
    "required": ["source", "description"],
    "additionalProperties": False,
}

response = track(client.messages.create(
    model=MODEL,
    max_tokens=1000,
    output_config={
        "effort": "low",
        "format": {"type": "json_schema", "schema": rule_schema},
    },
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule for S3 object uploads."}],
))

rule = json.loads(text_from_message(response))
print(rule)

assert set(rule) == {"source", "description"}

[input=253 output=63 cost=$0.00114 | spent $0.0056 of $0.10]
{'source': ['aws.s3'], 'description': 'EventBridge rule matching S3 Object Created events (detail-type: Object Created) to trigger on new object uploads to a bucket'}


## 6. A real use: pulling facts out of messy text (paid)

This is where structured outputs earn their keep: turning free text into data you can store in a database. The example below is a fictional support email.

The class uses a few more Pydantic features:

| Python | Meaning |
|---|---|
| `Literal["free", "pro", "enterprise"]` | Must be one of these exact values |
| `bool`, `int`, `list[str]` | The value must be this type |
| `Optional[str] = None` | The value may be missing |
| `Field(ge=1, le=5)` | The number must be from 1 to 5 |
| `Field(description="...")` | Tells Claude what the field means |

In [9]:
from typing import Literal, Optional

from pydantic import Field


class SupportTicket(BaseModel):
    customer_name: str
    email: str
    plan: Literal["free", "pro", "enterprise"]
    wants_demo: bool
    products_mentioned: list[str]
    company: Optional[str] = None
    urgency: int = Field(ge=1, le=5, description="1 is not urgent at all, 5 is an emergency")


email = """
Hi team, Sam here (sam@example.com). We're on your Pro plan and the export button in
Reports has been failing since this morning. Our board meeting is tomorrow and I need
those numbers! Also, could someone show us a demo of Dashboards while you're at it?
"""

response = track(client.messages.parse(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": f"Extract the details from this support email:\n\n{email}"}],
    output_format=SupportTicket,
))

ticket = response.parsed_output
print(ticket.model_dump_json(indent=2))

assert ticket.email == "sam@example.com"
assert ticket.plan == "pro"
assert ticket.wants_demo is True
assert ticket.company is None
assert 1 <= ticket.urgency <= 5

[input=679 output=66 cost=$0.00202 | spent $0.0096 of $0.10]
{
  "customer_name": "Sam",
  "email": "sam@example.com",
  "plan": "pro",
  "wants_demo": true,
  "products_mentioned": [
    "Reports",
    "Dashboards"
  ],
  "company": null,
  "urgency": 4
}


**What to notice**

- The email never mentions a company, so `company` is `None`. An optional field gives Claude an honest way to say "not given", which makes it less likely to invent something.
- `urgency` is Claude's judgment. The schema guarantees **a whole number from 1 to 5**, not that the number is right.
- The API doesn't support range rules like `ge` and `le` directly. `parse()` moves them into the field's description, then checks the result itself, and raises an error if a value breaks the rule.

## 7. Not just JSON: code and other formats (paid)

The course also uses prefill to get bare code or lists with no commentary. With structured outputs, you put that content in a field. Your code reads `answer.code`, and there's nothing around it to strip away.

To check the result is real Python, we parse it with `ast.parse`, without running it. Notebook 03 builds this kind of check into a full grader.

In [10]:
import ast


class PythonCode(BaseModel):
    code: str


response = track(client.messages.parse(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Write a Python function called reverse_string that reverses a string."}],
    output_format=PythonCode,
))

answer = response.parsed_output
print(answer.code)

ast.parse(answer.code)  # raises SyntaxError if this isn't valid Python
print("\nValid Python.")

[input=233 output=56 cost=$0.00103 | spent $0.0107 of $0.10]
def reverse_string(s):
    """Return the reverse of the given string."""
    return s[::-1]


Valid Python.


## 8. When structured outputs can still fail (paid)

The shape is guaranteed only when Claude finishes normally. Here we set `max_tokens` too low on purpose, so the JSON gets cut off.

In [11]:
from pydantic import ValidationError


class CountryList(BaseModel):
    countries: list[str]


try:
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=20,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": "List 30 countries."}],
        output_format=CountryList,
    ))
    print("No error. stop_reason:", response.stop_reason, "| parsed_output:", response.parsed_output)
except ValidationError as error:
    print("ValidationError: the JSON was cut off, so it couldn't be parsed.")
    print(error.errors()[0]["msg"])

ValidationError: the JSON was cut off, so it couldn't be parsed.
Invalid JSON: EOF while parsing a string at line 1 column 51


**What to notice**

- `parse()` raises an error when the output doesn't fit your class, so your code never receives a half-filled object. Catch `ValidationError` wherever this could happen.
- **A failed `parse()` is still billed, and `track()` doesn't count it**, because `track()` never receives a response. Here that's a tiny amount (20 output tokens), but it's one reason the Console spend limit matters.

| Situation | What happens | What to do |
|---|---|---|
| Hit `max_tokens` | `stop_reason` is `max_tokens`, and the JSON is incomplete | Raise `max_tokens`, leaving room for thinking too |
| Claude declines | `stop_reason` is `refusal`, and the text may not match your schema | Don't retry it unchanged |
| A number breaks a `Field` rule | `parse()` raises `ValidationError` | Catch it, then retry or handle it |
| An enum in different capitalization | The API doesn't guarantee capitalization of enum values | Use lowercase values, and compare without case where it matters |
| Using citations too | Citations can't be combined with structured outputs (400 error) | Choose one for that request |
| Unsupported schema feature, such as recursive schemas | 400 error that explains why | Simplify the schema |

**The biggest limit:** a schema guarantees the **shape** of an answer, not that it's **true**. Claude can fill a perfectly valid form with a wrong value. Finding out how often that happens is what the next notebook is about.

## Final spend

In [12]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0107 of its $0.10 budget.


## Takeaways

1. **Asking for JSON in a prompt is a request, not a guarantee.**
2. **Assistant prefill is gone** on current models. It returns a 400 error.
3. **Use `client.messages.parse()` with a Pydantic class.** You get a validated object in `response.parsed_output`.
4. **Use `output_config.format` with a raw schema** when you don't want Pydantic.
5. **Use `Optional`** for details that may be missing, so Claude isn't pushed to invent them.
6. **Catch `ValidationError`**, because cut-off output raises it, and that failed call is still billed.
7. **A schema guarantees shape, not truth.**

## Your turn

Each exercise costs about a cent or less.

- In section 4, add a `state: str` field to `EventBridgeRule`. What does Claude put in it?
- In section 6, remove `Optional` from `company`, so it becomes required. What does Claude put there now that the email doesn't mention a company?
- In section 6, change `plan` to `Literal["free", "basic"]`, so the real answer isn't an option. What happens?
- In section 8, try `max_tokens=500`. Does the error go away?

**Next:** [03 · Prompt evaluation](03_prompt_evaluation.ipynb): measuring how good a prompt really is, with test datasets and graders.